# FinAI Researcher Stack Verification & Walkthrough


### Services Covered:
1. **SageMaker Serverless Embeddings**: Generates vector representations (embeddings) of text.
2. **Amazon Bedrock (Moonshot Kimi)**: Interacts with the Kimi LLM to verify model endpoints.
3. **Web Research Lambda Function**: Simulates an end-to-end trigger for the Web Research agent (scrapes, analyzes, and index data).
4. **S3 Vectors Querying**: Retrieves query embeddings from SageMaker and queries our custom S3 Vectors VDB index to demonstrate semantic search.

In [ ]:
import json
import os
import boto3
from dotenv import load_dotenv
from botocore.config import Config

# Load environment variables from the project's .env file
load_dotenv()

region = os.getenv("DEFAULT_AWS_REGION", "us-east-1")
print(f"[SETUP] Loaded environment configuration.")
print(f"[SETUP] Target AWS Region: {region}")

# Get AWS Account details to locate regional resources
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]
print(f"[SETUP] Connected AWS Account ID: {account_id}")

[SETUP] Loaded environment configuration.
[SETUP] Target AWS Region: us-east-1
[SETUP] Connected AWS Account ID: 905418342208


## 1. Test SageMaker Serverless Embeddings

This section tests the serverless SageMaker embedding model (`finai-embedding-endpoint`) using the HuggingFace `all-MiniLM-L6-v2` model. This endpoint generates dense vectors (embeddings) representing the semantic meaning of the input text.

### Under the Hood:
- **Input**: A JSON dictionary `{"inputs": "<your text>"}`.
- **API Client**: `boto3.client('sagemaker-runtime')` invoking the serverless endpoint.
- **Output**: A list representing the dense vector of 384 dimensions (expected format `[[vector]]` or `[vector]`).

In [2]:
endpoint_name = os.getenv("SAGEMAKER_ENDPOINT", "finai-embedding-endpoint")
text_to_embed = "Test Indian equity research document content for semantic matching"

print("--- Testing SageMaker Serverless Embeddings ---")
print(f"Targeting SageMaker Endpoint: '{endpoint_name}' in region: '{region}'")
print(f"Input Text: \"{text_to_embed}\"\n")

sagemaker_client = boto3.client("sagemaker-runtime", region_name=region)
payload = {"inputs": text_to_embed}

print(f"[DEBUG] Constructing Request Payload: {json.dumps(payload)}")

try:
    response = sagemaker_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(payload),
    )
    
    # Extract and parse the response body
    response_body = response["Body"].read().decode("utf-8")
    result = json.loads(response_body)
    
    print("\n--- SageMaker Response Details ---")
    print(f"Response Content-Type: {response.get('ContentType')}")
    print(f"HTTP Status Code: {response.get('ResponseMetadata', {}).get('HTTPStatusCode')}")
    
    if isinstance(result, list):
        # Handle nested list formats that HuggingFace pipeline containers might return
        embedding = result[0][0] if isinstance(result[0][0], list) else result
        print(f"[PASS] Success! Embedding vector generated.")
        print(f"  Vector Length/Dimension: {len(embedding)} (Expected: 384)")
        print(f"  Sample Vector Values (First 5 dimensions): {embedding[:5]}...")
    else:
        print(f"[FAIL] Unexpected response payload type: {type(result)}")
        print(result)
except Exception as e:
    print(f"[FAIL] Failed to invoke SageMaker endpoint: {e}")

--- Testing SageMaker Serverless Embeddings ---
Targeting SageMaker Endpoint: 'finai-embedding-endpoint' in region: 'us-east-1'
Input Text: "Test Indian equity research document content for semantic matching"

[DEBUG] Constructing Request Payload: {"inputs": "Test Indian equity research document content for semantic matching"}

--- SageMaker Response Details ---
Response Content-Type: application/json
HTTP Status Code: 200
[PASS] Success! Embedding vector generated.
  Vector Length/Dimension: 384 (Expected: 384)
  Sample Vector Values (First 5 dimensions): [-0.04837629571557045, 0.06733988970518112, -0.5484427809715271, 0.08974839746952057, -0.5167555212974548]...


## 2. Test Amazon Bedrock (Moonshot Kimi LLM)

This section verifies that the Amazon Bedrock service can invoke the `moonshotai.kimi-k2.5` model.

### Under the Hood:
- **Input**: A standard messages payload (structured similarly to OpenAI's completion API: `{"messages": [{"role": "user", "content": "..."}], "max_tokens": 150}`).
- **API Client**: `boto3.client('bedrock-runtime')` invoking the model id.
- **Output**: A JSON payload containing generated completion choices under `.choices[0].message.content`.

In [3]:
model_id = os.getenv("BEDROCK_MODEL_ID", "moonshotai.kimi-k2.5")
bedrock_region = os.getenv("BEDROCK_REGION", "us-east-1")
prompt_message = "Hello! List 3 main stock exchanges in India."

print("--- Testing Bedrock Kimi Model Inference ---")
print(f"Targeting Bedrock Model ID: '{model_id}' in region: '{bedrock_region}'")
print(f"Prompt: \"{prompt_message}\"\n")

bedrock_client = boto3.client("bedrock-runtime", region_name=bedrock_region)
payload = {
    "messages": [
        {
            "role": "user",
            "content": prompt_message,
        }
    ],
    "max_tokens": 150,
}

print(f"[DEBUG] Constructing Request Payload: {json.dumps(payload)}")

try:
    response = bedrock_client.invoke_model(
        modelId=model_id,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(payload),
    )
    
    # Extract and parse response body
    response_body = json.loads(response.get("body").read().decode("utf-8"))
    print("\n--- Bedrock Raw Response Object ---")
    print(json.dumps(response_body, indent=2))
    
    content = response_body.get("choices", [{}])[0].get("message", {}).get("content", "")
    
    print("\n--- Extracted Response Content ---")
    if content:
        print("[PASS] Bedrock responded successfully!")
        safe_content = content.strip().encode("ascii", errors="replace").decode("ascii")
        print(f"\n{safe_content}\n")
    else:
        print("[FAIL] Bedrock returned empty response content.")
except Exception as e:
    print(f"[FAIL] Failed to invoke Bedrock model: {e}")

--- Testing Bedrock Kimi Model Inference ---
Targeting Bedrock Model ID: 'moonshotai.kimi-k2.5' in region: 'us-east-1'
Prompt: "Hello! List 3 main stock exchanges in India."

[DEBUG] Constructing Request Payload: {"messages": [{"role": "user", "content": "Hello! List 3 main stock exchanges in India."}], "max_tokens": 150}

--- Bedrock Raw Response Object ---
{
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": " Hello! Here are the 3 main stock exchanges in India:\n\n1. **National Stock Exchange (NSE)** \u2013 Located in Mumbai, it is the largest stock exchange in India by trading volume and market capitalization.\n\n2. **Bombay Stock Exchange (BSE)** \u2013 Also based in Mumbai, it is Asia's oldest stock exchange (established in 1875) and one of the world's fastest exchanges.\n\n3. **Metropolitan Stock Exchange (MSEI)** \u2013 Headquartered in Mumbai, it focuses on newer financial instruments and operates

## 3. Web Research using Triggering Research Lambda

We will now test the end-to-end Researcher Agent by invoking the `finai-researcher` Lambda directly. This triggers the web search, parses webpages, calls Kimi to analyze, extracts embeddings using SageMaker, and writes chunks to S3 Vectors DB.

### Under the Hood:
- **Input**: A mock Lambda Function URL Event payload containing the `topic` inside the POST body request.
- **API Client**: `boto3.client('lambda')` with a higher read timeout (180s) to allow scraping, AI parsing, and indexing tasks to execute synchronously.
- **Output**: A JSON HTTP response structure enclosing the AI's research report / summary.

In [4]:
lambda_name = "finai-researcher"
research_topic = "Tata Consultancy Services (TCS)"

print("--- Testing End-to-End Researcher Lambda ---")
print(f"Target Lambda Function Name: '{lambda_name}'")
print(f"Target Topic: '{research_topic}'\n")

# Configure custom timeouts because agent web research & ingestion tasks require extra processing time
lambda_client = boto3.client(
    "lambda",
    region_name=region,
    config=Config(read_timeout=180, connect_timeout=10),
)

# Simulate a Lambda Function URL v2 Request payload
payload = {
    "version": "2.0",
    "rawPath": "/research",
    "requestContext": {"http": {"method": "POST", "path": "/research"}},
    "headers": {"content-type": "application/json"},
    "body": json.dumps({"topic": research_topic}),
}

print(f"[DEBUG] Constructing Lambda Event Payload:\n{json.dumps(payload, indent=2)}\n")
print("[RUNNING] Invoking Researcher Lambda... (This will take 30-60 seconds for scraping and ingestion)")

try:
    response = lambda_client.invoke(
        FunctionName=lambda_name,
        Payload=json.dumps(payload),
    )
    
    result_payload = json.loads(response["Payload"].read().decode("utf-8"))
    
    print("\n--- Lambda Response Status ---")
    print(f"Lambda Service Status Code: {response.get('StatusCode')}")
    
    if "body" in result_payload:
        body_content = result_payload["body"]
        if isinstance(body_content, str):
            body_content = json.loads(body_content)
            
        # Check for FastAPI / application errors
        if isinstance(body_content, dict) and "detail" in body_content:
            print(f"[FAIL] Lambda app returned error: {body_content['detail']}")
        else:
            print("[PASS] E2E Success! Researcher Lambda execution completed.")
            body_str = str(body_content)
            safe_summary = body_str[:500].encode("ascii", errors="replace").decode("ascii")
            print(f"\nSample of Agent Analysis Summary (First 500 chars):\n{safe_summary}...\n")
    else:
        print(f"[FAIL] Unexpected Lambda response structure: {result_payload}")
except Exception as e:
    print(f"[FAIL] Failed to invoke Lambda: {e}")

--- Testing End-to-End Researcher Lambda ---
Target Lambda Function Name: 'finai-researcher'
Target Topic: 'Tata Consultancy Services (TCS)'

[DEBUG] Constructing Lambda Event Payload:
{
  "version": "2.0",
  "rawPath": "/research",
  "requestContext": {
    "http": {
      "method": "POST",
      "path": "/research"
    }
  },
  "headers": {
    "content-type": "application/json"
  },
  "body": "{\"topic\": \"Tata Consultancy Services (TCS)\"}"
}

[RUNNING] Invoking Researcher Lambda... (This will take 30-60 seconds for scraping and ingestion)

--- Lambda Response Status ---
Lambda Service Status Code: 200
[PASS] E2E Success! Researcher Lambda execution completed.

Sample of Agent Analysis Summary (First 500 chars):
 ## Tata Consultancy Services (TCS) - Quick Summary

**Current Price:** ?2,088 (+0.96% today)

### Key Metrics
| Metric | Value |
|--------|-------|
| Market Cap | ?7.57T (~$90B) |
| P/E Ratio | 15.35x |
| Dividend Yield | 3.07% |
| 52-Week High/Low | ?3,435 / ?1,977 |
| E

## 4. Retrieve Embeddings from SageMaker and Query S3 Vectors

To demonstrate and verify the vector database retrieval under the hood, we query our **S3 Vectors DB** manually. We do this by:
1. Converting a search query (e.g. "Tata Consultancy Services (TCS) financial overview") into a vector representation by invoking our SageMaker endpoint.
2. Querying the custom `s3vectors` client with the embedding to retrieve conceptual matches.

### Under the Hood:
- **Embedding Retrieval**: Call SageMaker to get a 384-dimensional vector for our query text.
- **Index Retrieval**: Invoke the Custom AWS resource `s3vectors` via `query_vectors` pointing to our vector S3 bucket (`finai-vectors-<account-id>`) and index (`financial-research`).
- **Result Parsing**: Parse and calculate similarity scores (`1.0 - distance`), displaying the matched segments.

In [5]:
bucket_name = f"finai-vectors-{account_id}"
index_name = "financial-research"
search_query = "Tata Consultancy Services (TCS) financial overview"

print("--- Testing S3 Vectors Retrieval (Semantic Search) ---")
print(f"Target Vector S3 Bucket: '{bucket_name}'")
print(f"Index Name: '{index_name}'")
print(f"Query String: \"{search_query}\"\n")

# Step A: Call SageMaker to generate embedding for our query
print("[STEP A] Generating Query Embedding from SageMaker...")
sm_payload = {"inputs": search_query}

try:
    sm_response = sagemaker_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(sm_payload),
    )
    sm_result = json.loads(sm_response["Body"].read().decode("utf-8"))
    
    # Extract vector and normalize
    query_embedding = sm_result[0] if isinstance(sm_result[0], list) else sm_result
    if isinstance(query_embedding[0], list):
        query_embedding = query_embedding[0]
        
    print(f"[PASS] Generated query embedding vector of length: {len(query_embedding)}")
    
    # Step B: Query S3 Vectors DB using the query embedding vector
    print("\n[STEP B] Invoking s3vectors Query API...")
    s3vectors_client = boto3.client("s3vectors", region_name=region)
    
    search_response = s3vectors_client.query_vectors(
        vectorBucketName=bucket_name,
        indexName=index_name,
        queryVector={"float32": query_embedding},
        topK=3,
        returnDistance=True,
        returnMetadata=True,
    )
    
    vectors = search_response.get("vectors", [])
    print(f"Response returned {len(vectors)} semantic matches.")
    
    print("\n--- Matches Extracted ---")
    if vectors:
        print(f"[PASS] Successfully retrieved matches from S3 Vectors database!")
        for i, vec in enumerate(vectors, 1):
            metadata = vec.get("metadata", {})
            # Compute similarity score: similarity = 1 - cosine distance
            score = 1.0 - vec.get("distance", 0.0)
            text_preview = metadata.get("text", "")
            if len(text_preview) > 200:
                text_preview = text_preview[:200] + "..."
            
            # Ensure safe printable format
            safe_preview = text_preview.encode("ascii", errors="replace").decode("ascii")
            
            print(f"\nMatch #{i} (Similarity Score: {score:.4f}):")
            print(f"  - Source URL/File: {metadata.get('source', 'N/A')}")
            print(f"  - Associated Topic: {metadata.get('topic', 'N/A')}")
            print(f"  - Text Segment: {safe_preview}")
    else:
        print("[WARNING] Query returned 0 matches. Please check if the Lambda research invocation successfully indexed pages to this index.")

except Exception as e:
    print(f"[FAIL] S3 Vectors query execution failed: {e}")

--- Testing S3 Vectors Retrieval (Semantic Search) ---
Target Vector S3 Bucket: 'finai-vectors-905418342208'
Index Name: 'financial-research'
Query String: "Tata Consultancy Services (TCS) financial overview"

[STEP A] Generating Query Embedding from SageMaker...
[PASS] Generated query embedding vector of length: 384

[STEP B] Invoking s3vectors Query API...
Response returned 1 semantic matches.

--- Matches Extracted ---
[PASS] Successfully retrieved matches from S3 Vectors database!

Match #1 (Similarity Score: 0.7883):
  - Source URL/File: N/A
  - Associated Topic: TCS Analysis Jul 04
  - Text Segment: # Tata Consultancy Services (TCS) - Investment Analysis
**Date:** July 4, 2026

## Key Metrics
- **Current Price:** ?2,088.00 (+0.96%, +?19.90 today)
- **Market Cap:** ?7.57 Trillion (~$90B USD)
- **P...
